# Enhanced S3 to COG Converter with Automatic AWS Authentication

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Support for multiple AWS authentication methods**

Author: Kyle Lesinger (Enhanced version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
import rioxarray as rxr
import s3fs
import fsspec
from rasterio.warp import calculate_default_transform, reproject, Resampling
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import re

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.39.11


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked,
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links

[drcs_activations OLD Directory](https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/)

[VEDA docs for file naming conventions](https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html)

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:
EVENT_NAME = '202507_Flood_TX'
#old name
#under drcs_activations
PRODUCT_NAME = 'uavsar'

PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [6]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name='nasa-disasters', verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name='nasa-disasters', verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, 'nasa-disasters', PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

✅ S3 client initialized successfully
   Found 68 accessible buckets
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 14 .tif files in the S3 bucket.


['drcs_activations/202507_Flood_TX/uavsar/Classified UAVSAR Im_1.tif',
 'drcs_activations/202507_Flood_TX/uavsar/Classified UAVSAR Im_2.tif',
 'drcs_activations/202507_Flood_TX/uavsar/Classified UAVSAR Imagery.tif',
 'drcs_activations/202507_Flood_TX/uavsar/ClassifiedUAVSARI_CopyRas.tif',
 'drcs_activations/202507_Flood_TX/uavsar/colora_11802_25023_006_250709_L090_UNet_class.tif',
 'drcs_activations/202507_Flood_TX/uavsar/colora_11802_25023_006_250709_L090_UNet_predicted_score.tif',
 'drcs_activations/202507_Flood_TX/uavsar/flight25023_mosaic_UNet_class.tif',
 'drcs_activations/202507_Flood_TX/uavsar/flight25023_mosaic_UNet_class_grayscale.tif',
 'drcs_activations/202507_Flood_TX/uavsar/guadal_11013_25023_002_250709_L090_UNet_class.tif',
 'drcs_activations/202507_Flood_TX/uavsar/guadal_11013_25023_002_250709_L090_UNet_predicted_score.tif',
 'drcs_activations/202507_Flood_TX/uavsar/sangab_26918_25023_007_250709_L090_UNet_class.tif',
 'drcs_activations/202507_Flood_TX/uavsar/sangab_26918

## Load TIF Files from DRCS Data
### This may assist with diagnosing any issues that occur if no files are found in the code block above

This cell loads the pre-analyzed DRCS activation data from `drcs_activations_tif_files.json` which contains a complete inventory of all .tif files in the NASA Disasters S3 bucket.

The code will:
1. Load the JSON file containing the file inventory
2. Parse the `PATH_OLD` variable to find the corresponding directory
3. Extract all .tif filenames from that directory
4. Store them in `files_to_process` for later use

In [38]:
# # Load the pre-analyzed DRCS TIF files data using imported functions
# # The JSON path is relative to the notebook location
# json_path = Path('../../s3-crawler/drcs_activations_tif_files.json')

# # Load DRCS data
# drcs_data = load_drcs_data(json_path)

# if drcs_data:
#     # Get TIF files from the specified PATH_OLD using the imported function
#     tif_files = get_tif_files_from_path(PATH_OLD, drcs_data, DIR_OLD_BASE)
    
#     if tif_files:
#         print(f"\n📁 Found {len(tif_files)} .tif files in {PATH_OLD}:")
#         print("\nFirst 10 files:")
#         for i, file in enumerate(tif_files[:10], 1):
#             print(f"  {i:2d}. {file}")
#         if len(tif_files) > 10:
#             print(f"  ... and {len(tif_files) - 10} more files")
        
#         # Get files with full paths using the imported function
#         files_to_process = get_files_with_full_paths(PATH_OLD, drcs_data, DIR_OLD_BASE, json_path)
#         print(f"\n✅ Files ready for processing. Stored in 'files_to_process' variable.")
#     else:
#         print(f"\n❌ No files found. Please check the PATH_OLD variable.")
#         files_to_process = []
# else:
#     print(f"\n❌ Could not load DRCS data.")
#     files_to_process = []

# files_to_process

In [39]:
# # Example: List available activation events using the imported function
# print("📂 Available activation events in DRCS data:")
# events = list_available_directories('drcs_activations', drcs_data, json_path)

# # Show first 10 events
# for event in events[:10]:
#     print(f"  - {event}")
# if len(events) > 10:
#     print(f"  ... and {len(events) - 10} more events")

# # Example: List subdirectories for a specific event
# print(f"\n📁 Subdirectories in {EVENT_NAME}:")
# subdirs = list_available_directories(f'drcs_activations/{EVENT_NAME}', drcs_data, json_path)
# for subdir in subdirs:
#     print(f"  - {subdir}")

In [6]:
config_uavsar = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}


## Configure bucket and paths (no need to create session manually)

In [8]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    _
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

In [10]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [14]:
def convert_date(date_str):
    """
    Convert to YYYY-MM-DD.
    
    Args:
        datetime_str: String like '20250731'
    
    Returns:
        String like '2025-07-31'
    """
    # Extract components
    year = date_str[0:4]
    month = date_str[4:6]
    day = date_str[6:8]
    
    # Format with dashes and colons, add Z for UTC
    return f"{year}-{month}-{day}"

# Test
date_str = '20250731'
result = convert_date(date_str)
print(result)

2025-07-31


In [16]:
def create_cog_filename(f, EVENT_NAME):
    """Create COG filename for NDVI files, handling single or dual dates."""
    # Extract directory, filename, and extension
    directory, filename = os.path.split(f)
    stem, ext = os.path.splitext(filename)

    # Find all 8-digit date patterns
    dates = re.findall(r"\d{8}", stem)

    # Remove dates from the stem
    stem_clean = re.sub(r"_?\d{8}", "", stem)

    if len(dates) == 1:
        # Single date
        cog_filename = f"{EVENT_NAME}_{stem_clean}_{convert_date(dates[0])}_day.tif"
    elif len(dates) == 2:
        # Two dates → comparison format
        cog_filename = f"{EVENT_NAME}_{stem_clean}_c{convert_date(dates[0])}_{convert_date(dates[1])}_day.tif"
    else:
        raise ValueError(f"Unexpected number of dates in filename: {filename}")

    return cog_filename


filter_str = 'NDVI'

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:


In [20]:
import os
import re
from datetime import datetime

def _to_iso_from_8(s):
    # s like "20250709" -> "2025-07-09"
    return datetime.strptime(s, "%Y%m%d").strftime("%Y-%m-%d")

def _to_iso_from_6_yymmdd(s):
    # s like "250709" -> "20250709" -> "2025-07-09"
    ymd = "20" + s
    return datetime.strptime(ymd, "%Y%m%d").strftime("%Y-%m-%d")

def _to_ym_from_6_yyyymm(s):
    # s like "202507" -> "2025-07"
    return datetime.strptime(s, "%Y%m").strftime("%Y-%m")

def create_cog_filename(f, EVENT_NAME, default_month="2025-07"):
    """
    Create COG filename:
      - 8-digit YYYYMMDD -> daily (YYYY-MM-DD_day.tif)
      - 6-digit starting with '25'  -> treat as YYMMDD -> daily (2025-MM-DD_day.tif)
      - 6-digit starting with '20'  -> treat as YYYYMM -> monthly (YYYY-MM_monthly.tif)
      - no date -> default monthly (default_month_monthly.tif)
    Replaces spaces in the stem with hyphens.
    """
    directory, filename = os.path.split(f)
    stem, ext = os.path.splitext(filename)

    # combined pattern: 8-digit YYYYMMDD, 6-digit starting with 25 (YYMMDD), 6-digit starting with 20 (YYYYMM)
    pattern = re.compile(
        r'(?P<Y8>(20\d{6}))'    # 8-digit YYYYMMDD (starts with 20)
        r'|(?P<YY6>(25\d{4}))'  # 6-digit starting with 25 -> YYMMDD
        r'|(?P<YM6>(20\d{4}))'  # 6-digit starting with 20 -> YYYYMM (monthly)
    )

    # collect matches in order of appearance
    matches = []
    for m in pattern.finditer(stem):
        if m.group('Y8'):
            matches.append(('Y8', m.group('Y8'), m.start(), m.end()))
        elif m.group('YY6'):
            matches.append(('YY6', m.group('YY6'), m.start(), m.end()))
        elif m.group('YM6'):
            # ensure we don't double-capture the 8-digit as two 6-digit matches:
            matches.append(('YM6', m.group('YM6'), m.start(), m.end()))

    # remove matched date tokens from stem (remove optional leading _ or - as well)
    stem_clean = stem
    # reverse order so replacements don't shift earlier indices
    for _, val, _, _ in reversed(matches):
        stem_clean = re.sub(r'[_-]?' + re.escape(val), '', stem_clean, count=1)

    # normalize: replace spaces with hyphens, collapse multiple hyphens, strip separators
    stem_clean = stem_clean.replace(" ", "-")
    stem_clean = re.sub(r'[-_]{2,}', '-', stem_clean)
    stem_clean = stem_clean.strip("-_")

    # convert matched tokens to iso strings and record their kind
    date_iso = []
    date_kind = []  # 'day' or 'month'
    for kind, val, _, _ in matches:
        if kind == 'Y8':
            date_iso.append(_to_iso_from_8(val))
            date_kind.append('day')
        elif kind == 'YY6':
            date_iso.append(_to_iso_from_6_yymmdd(val))
            date_kind.append('day')
        elif kind == 'YM6':
            date_iso.append(_to_ym_from_6_yyyymm(val))
            date_kind.append('month')

    # build filename
    if not date_iso:
        cog_filename = f"{EVENT_NAME}_{stem_clean}_{default_month}_monthly.tif"
    elif len(date_iso) == 1:
        if date_kind[0] == 'day':
            cog_filename = f"{EVENT_NAME}_{stem_clean}_{date_iso[0]}_day.tif"
        else:
            cog_filename = f"{EVENT_NAME}_{stem_clean}_{date_iso[0]}_monthly.tif"
    elif len(date_iso) == 2:
        # both daily -> daily comparison
        if date_kind[0] == 'day' and date_kind[1] == 'day':
            cog_filename = f"{EVENT_NAME}_{stem_clean}_c{date_iso[0]}_{date_iso[1]}_day.tif"
        # both monthly -> monthly comparison
        elif date_kind[0] == 'month' and date_kind[1] == 'month':
            cog_filename = f"{EVENT_NAME}_{stem_clean}_c{date_iso[0]}_{date_iso[1]}_monthly.tif"
        else:
            # mixed -> prefer day-suffix (put both iso strings, use _day.tif)
            cog_filename = f"{EVENT_NAME}_{stem_clean}_c{date_iso[0]}_{date_iso[1]}_day.tif"
    else:
        raise ValueError(f"More than two date matches in filename: {filename}")

    return cog_filename

filter_str = ''

# Test functions
print("Testing WM filename:")
filter_ = [i for i in keys if filter_str in i]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202507_Flood_TX_Classified-UAVSAR-Im_1_2025-07_monthly.tif
  202507_Flood_TX_Classified-UAVSAR-Im_2_2025-07_monthly.tif
  202507_Flood_TX_Classified-UAVSAR-Imagery_2025-07_monthly.tif
  202507_Flood_TX_ClassifiedUAVSARI_CopyRas_2025-07_monthly.tif
  202507_Flood_TX_colora_11802_25023_006_L090_UNet_class_2025-07-09_day.tif
  202507_Flood_TX_colora_11802_25023_006_L090_UNet_predicted_score_2025-07-09_day.tif
  202507_Flood_TX_flight25023_mosaic_UNet_class_2025-07_monthly.tif
  202507_Flood_TX_flight25023_mosaic_UNet_class_grayscale_2025-07_monthly.tif
  202507_Flood_TX_guadal_11013_25023_002_L090_UNet_class_2025-07-09_day.tif
  202507_Flood_TX_guadal_11013_25023_002_L090_UNet_predicted_score_2025-07-09_day.tif
  202507_Flood_TX_sangab_26918_25023_007_L090_UNet_class_2025-07-09_day.tif
  202507_Flood_TX_sangab_26918_25023_007_L090_UNet_predicted_score_2025-07-09_day.tif
  202507_Flood_TX_sangab_30412_25023_005_L090_UNet_class_2025-07-09_day.tif
  202507_Flood_TX_san

In [21]:
# # Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = filter_str, 
                                rename_func = create_cog_filename, 
                                target_dir = "UAVSAR", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202507_Flood_TX_Classified-UAVSAR-Im_1_2025-07_monthly.tif
  202507_Flood_TX_Classified-UAVSAR-Im_2_2025-07_monthly.tif
  202507_Flood_TX_Classified-UAVSAR-Imagery_2025-07_monthly.tif
  202507_Flood_TX_ClassifiedUAVSARI_CopyRas_2025-07_monthly.tif
  202507_Flood_TX_colora_11802_25023_006_L090_UNet_class_2025-07-09_day.tif
  202507_Flood_TX_colora_11802_25023_006_L090_UNet_predicted_score_2025-07-09_day.tif
  202507_Flood_TX_flight25023_mosaic_UNet_class_2025-07_monthly.tif
  202507_Flood_TX_flight25023_mosaic_UNet_class_grayscale_2025-07_monthly.tif
  202507_Flood_TX_guadal_11013_25023_002_L090_UNet_class_2025-07-09_day.tif
  202507_Flood_TX_guadal_11013_25023_002_L090_UNet_predicted_score_2025-07-09_day.tif
  202507_Flood_TX_sangab_26918_25023_007_L090_UNet_class_2025-07-09_day.tif
  202507_Flood_TX_sangab_26918_25023_007_L090_UNet_predicted_score_2025-07-09_day.tif
  202507_Flood_TX_sangab_30412_25023_005_L090_UNet_class_2025-07-09_day.tif
  202507_Flood_TX_sanga

/srv/conda/envs/notebook/lib/python3.12/site-packages/rio_cogeo/cogeo.py:226: NodataAlphaMaskWarning: Input dataset has both a nodata value and internal alpha/mask band. Nodata value will be prioritized.
  warnings.warn(
Reading input: /tmp/tmpk_i1wtp0_temp.tif



   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpcbc_pg4w.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/202507_Flood_TX_Classified-UAVSAR-Im_1_2025-07_monthly.tif
   [MEMORY] Final: 1117.2 MB (Change: +814.4 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202507_Flood_TX_Classified-UAVSAR-Im_1_2025-07_monthly.tif

[2/14] Processing: drcs_activations/202507_Flood_TX/uavsar/Classified UAVSAR Im_2.tif
   Output filename: 202507_Flood_TX_Classified-UAVSAR-Im_2_2025-07_monthly.tif
   [MEMORY] Initial: 1117.2 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0, max=255, center sample non-zero=92/1000000
            Estimated data coverage: 3.1% (from distr

...

   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/202507_Flood_TX_sangab_30412_25023_005_L090_UNet_class_2025-07-09_day.tif
   [MEMORY] Final: 1425.5 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202507_Flood_TX_sangab_30412_25023_005_L090_UNet_class_2025-07-09_day.tif

[14/14] Processing: drcs_activations/202507_Flood_TX/uavsar/sangab_30412_25023_005_250709_L090_UNet_predicted_score.tif
   Output filename: 202507_Flood_TX_sangab_30412_25023_005_L090_UNet_predicted_score_2025-07-09_day.tif
   [MEMORY] Initial: 1425.5 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [VERIFY] Checking reprojected data...
   [VERIFY] Band 1: min=0.0, max=0.9

Reading input: /tmp/tmpmo3vuhno_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpyxvflgzn.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/202507_Flood_TX_sangab_30412_25023_005_L090_UNet_predicted_score_2025-07-09_day.tif
   [MEMORY] Final: 1428.3 MB (Change: +2.7 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202507_Flood_TX_sangab_30412_25023_005_L090_UNet_predicted_score_2025-07-09_day.tif

✅ Batch processing complete: 14 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/UAVSAR/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/UAVSAR/files_converted.csv
📁 COGs saved locally to: output/202507_Flood_TX

📊 BATCH PROCESSING SUMMARY
Total files processed: 14
Successful: 14
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-20T17:32:43.464346


## Define COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with proper CRS and caching.

In [19]:
def convert_to_proper_CRS_and_cogify(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
    """
    Convert a file to Cloud Optimized GeoTIFF with proper CRS.
    
    This function includes:
    - Download caching to avoid re-downloading files
    - CRS reprojection to EPSG:4326
    - COG validation before upload
    - Upload to S3
    - Smart nodata value handling based on data type
    """
    s3_key = f"{cog_data_prefix}/{cog_filename}"
    reproject_filename = f"reproj/{cog_filename}"
    
    # Create necessary directories
    os.makedirs("reproj", exist_ok=True)
    
    # Create data_download directory for caching
    data_download_dir = "data_download"
    os.makedirs(data_download_dir, exist_ok=True)
    
    # Create subdirectory structure to match S3 path
    s3_path_parts = name.split('/')
    local_subdir = os.path.join(data_download_dir, *s3_path_parts[:-1])
    os.makedirs(local_subdir, exist_ok=True)
    
    # Local path for the downloaded file (persistent storage)
    local_download_path = os.path.join(data_download_dir, name)
    
    # Temporary file for processing
    temp_input_file = f"temp_{os.path.basename(name)}"

    try:
        # Check if file already exists locally
        if os.path.exists(local_download_path):
            print(f"   [CACHE HIT] Using cached file: {local_download_path}")
            import shutil
            shutil.copy(local_download_path, temp_input_file)
        else:
            # Download the file from S3
            print(f"   [DOWNLOAD] Downloading from S3...")
            s3_client.download_file(config["raw_data_bucket"], name, local_download_path)
            print(f"   [DOWNLOAD] ✅ Saved to cache")
            import shutil
            shutil.copy(local_download_path, temp_input_file)
        
        # Reproject to EPSG:4326
        print(f"   [REPROJECT] Converting to EPSG:4326...")
        with rasterio.open(temp_input_file) as src:
            dst_crs = "EPSG:4326"
            
            # Check if reprojection is needed
            if src.crs and src.crs.to_string() == dst_crs:
                print(f"   [REPROJECT] Already in {dst_crs}, skipping reprojection")
                import shutil
                shutil.copy(temp_input_file, reproject_filename)
            else:
                transform, width, height = calculate_default_transform(
                    src.crs, dst_crs, src.width, src.height, *src.bounds
                )
                kwargs = src.meta.copy()
                kwargs.update({
                    "driver": "COG",
                    "compress": "DEFLATE",
                    "crs": dst_crs,
                    "transform": transform,
                    "width": width,
                    "height": height
                })

                with rasterio.open(reproject_filename, "w", **kwargs) as dst:
                    for band_idx in range(1, src.count + 1):
                        reproject(
                            source=rasterio.band(src, band_idx),
                            destination=rasterio.band(dst, band_idx),
                            src_transform=src.transform,
                            src_crs=src.crs,
                            dst_transform=transform,
                            dst_crs=dst_crs,
                            resampling=Resampling.nearest,
                            wrapdateline=True
                        )

        # COGify & upload
        print(f"   [COGIFY] Creating COG...")
        ds = rxr.open_rasterio(reproject_filename)
        
        # Handle coordinate naming
        if "y" in ds.dims and "x" in ds.dims:
            ds = ds.rename({"y": "lat", "x": "lon"})
            ds.rio.set_spatial_dims("lon", "lat", inplace=True)
        
        # Smart nodata value handling based on data type
        print(f"   [NODATA] Data type: {ds.dtype}")
        if ds.dtype == 'uint8':
            # For RGB images (uint8), use 0 as nodata (black pixels)
            nodata_value = 0
            print(f"   [NODATA] Using nodata value {nodata_value} for uint8 data")
        elif ds.dtype == 'uint16':
            # For uint16, use 0 as nodata
            nodata_value = 0
            print(f"   [NODATA] Using nodata value {nodata_value} for uint16 data")
        else:
            # For float32, int16, int32, etc., use -9999
            nodata_value = -9999
            print(f"   [NODATA] Using nodata value {nodata_value} for {ds.dtype} data")
        
        ds.rio.write_nodata(nodata_value, inplace=True)

        with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
            tmp_name = tmp.name
            ds.rio.to_raster(tmp_name, **COG_PROFILE)
            
            # Validate COG
            print(f"   [VALIDATE] Checking COG validity...")
            is_valid_cog, validation_details = validate_cog(tmp_name)
            
            if is_valid_cog:
                print(f"   [VALIDATE] ✅ Valid COG")
            else:
                print(f"   [VALIDATE] ⚠️ COG validation warnings")
                critical_errors = [e for e in validation_details['errors'] if 'Invalid driver' in e]
                if critical_errors:
                    raise ValueError(f"Critical COG validation failed")
            
            # Upload to S3
            print(f"   [UPLOAD] Uploading to S3...")
            s3_client.upload_file(
                Filename=tmp_name,
                Bucket=cog_data_bucket,
                Key=s3_key
            )
            print(f"   [SUCCESS] ✅ Uploaded to s3://{cog_data_bucket}/{s3_key}")
            
            # Save locally if specified
            if local_output_dir:
                os.makedirs(local_output_dir, exist_ok=True)
                local_path = os.path.join(local_output_dir, cog_filename)
                import shutil
                shutil.copy(tmp_name, local_path)
            
    except Exception as e:
        print(f"   [ERROR] Failed: {str(e)}")
        raise
            
    finally:
        # Clean up temporary files
        for temp_file in [temp_input_file, reproject_filename]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        if 'tmp_name' in locals() and os.path.exists(tmp_name):
            os.remove(tmp_name)

print("✅ COG conversion function defined with smart nodata handling")

✅ COG conversion function defined with smart nodata handling


In [21]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 0
  - Total size: 0.00 GB


(0, 0)

# Process RGB files

In [12]:
for i in keys:
    head = s3_client.head_object(Bucket=uavsar_bucket['bucket_name'], Key=i)
    print("File size (MB):", head["ContentLength"] / 1024 / 1024)

File size (MB): 233.80956363677979
File size (MB): 233.80956363677979
File size (MB): 935.0884962081909
File size (MB): 58.48189353942871
File size (MB): 1.2417116165161133
File size (MB): 13.682003021240234
File size (MB): 2.865720748901367
File size (MB): 1.301187515258789
File size (MB): 0.8691291809082031
File size (MB): 12.49346923828125
File size (MB): 0.6113147735595703
File size (MB): 15.404109954833984
File size (MB): 1.0707836151123047
File size (MB): 11.175731658935547


In [22]:
## Process files using batch processing function

"""print("📊 File categorization:")
print(f"  - Water mask files: {len(water_mask)}")
print(f"  - RGB files: {len(rgb)}")
print(f"  - Water mask diff files: {len(water_mask_diff)}")
print(f"  - Total files: {len(keys)}")"""

# Initialize combined results DataFrame
all_files_processed = pd.DataFrame()

config = config_uavsar #change here (e.g., the bucket info

# Process water mask files


wm_results = process_file_batch(
    file_list=keys, #change here )e/g/ l1,l2,l3
    s3_client=s3_client,
    config=config,
    filename_creator_func=create_cog_filename, #change here
    processing_func=convert_to_proper_CRS_and_cogify,
    event_name=EVENT_NAME,
    save_metadata=True,
    save_csv=True,
    verbose=True
)
all_files_processed = pd.concat([all_files_processed, wm_results], ignore_index=True)



# Print overall summary
print_batch_summary(all_files_processed)

✅ Local output directory ready: output/202507_Flood_TX

[1/14] Processing: drcs_activations/202507_Flood_TX/uavsar/Classified UAVSAR Im_1.tif
   Output filename: 202507_Flood_TX_Classified_UAVSAR_Im_1_202507.tif
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326...
   [REPROJECT] Already in EPSG:4326, skipping reprojection
   [COGIFY] Creating COG...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/UAVSAR/202507_Flood_TX_Classified_UAVSAR_Im_1_202507.tif
   ✅ Generated and saved COG: 202507_Flood_TX_Classified_UAVSAR_Im_1_202507.tif

[2/14] Processing: drcs_activations/202507_Flood_TX/uavsar/Classified UAVSAR Im_2.tif
   Output filename: 202507_Flood_TX_Classified_UAVSAR_Im_2_202507.tif
   [DOWNLOAD] Downloading from S3...
   [

In [23]:
# Display final results
print(f"\n📊 Final Processing Results:")
print(f"Total files processed: {len(all_files_processed)}")
print(f"\nProcessed files DataFrame:")
all_files_processed


📊 Final Processing Results:
Total files processed: 14

Processed files DataFrame:


,file_name,COGs_created
0,drcs_activations/202507_Flood_TX/uavsar/Classi...,202507_Flood_TX_Classified_UAVSAR_Im_1_202507.tif
1,drcs_activations/202507_Flood_TX/uavsar/Classi...,202507_Flood_TX_Classified_UAVSAR_Im_2_202507.tif
2,drcs_activations/202507_Flood_TX/uavsar/Classi...,202507_Flood_TX_Classified_UAVSAR_Imagery_2025...
3,drcs_activations/202507_Flood_TX/uavsar/Classi...,202507_Flood_TX_ClassifiedUAVSARI_CopyRas_2025...
4,drcs_activations/202507_Flood_TX/uavsar/colora...,202507_Flood_TX_colora_11802_25023_006_L090_UN...
5,drcs_activations/202507_Flood_TX/uavsar/colora...,202507_Flood_TX_colora_11802_25023_006_L090_UN...
6,drcs_activations/202507_Flood_TX/uavsar/flight...,202507_Flood_TX_flight25023_mosaic_UNet_class_...
7,drcs_activations/202507_Flood_TX/uavsar/flight...,202507_Flood_TX_flight25023_mosaic_UNet_class_...
8,drcs_activations/202507_Flood_TX/uavsar/guadal...,202507_Flood_TX_guadal_11013_25023_002_L090_UN...
9,drcs_activations/202507_Flood_TX/uavsar/guadal...,202507_Flood_TX_guadal_11013_25023_002_L090_UN...


## Check STATUS
[Disasters Bucket](https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/)